# Analyze and plot LSCALE output

This is meant as a replacement for the X11 output of the original lscale.

In [ ]:
%matplotlib inline
import glob
import os
from pathlib import Path
import numpy as np
from math import pi, sqrt
from importlib import reload
import matplotlib.pyplot as plt
from matplotlib import pyplot
import pandas as pd
import gemmi
from scipy.optimize import curve_fit
import sys

In [ ]:
from ess.nmx.mtz_loader import (
    MTZFilepath, read_in_files, IntensityColumnName, SigmaIntensityColumnName, LambdaColumnName, HKLColumnName,
    SpaceGroupDesc, SpaceGroup, HKLEQ, LAMBDABinned, get_lambda_bin,
    get_reciprocal_asu, get_space_group,
    ReferenceNumber,
    get_hkl_eq,
    ReferenceGroup, get_reference_group,
)
import sciline as sl


pipeline = sl.Pipeline(
    providers = [
        read_in_files,get_reciprocal_asu, get_space_group,
        get_hkl_eq,
        get_lambda_bin,
        get_reference_group
        ],
    params={
        MTZFilepath: MTZFilepath('/Users/sunyoungyoo/ESS/pyscale/assets/test_large/TIM_ap2-2_3h_phi_157.mtz1'),
        IntensityColumnName: IntensityColumnName('I'),
        SigmaIntensityColumnName: SigmaIntensityColumnName('SIGI'),
        LambdaColumnName: LambdaColumnName('LAMBDA'),
        HKLColumnName: HKLColumnName('hkl'),
        SpaceGroupDesc: SpaceGroupDesc('C 1 2 1'),
        ReferenceNumber: ReferenceNumber(250),  # Should have a default value, middle of the groups.
    }
)

pipeline

In [ ]:
pipeline.visualize(LAMBDABinned)

In [ ]:
from ess.nmx.mtz_loader import MTZFile

df1, ref = pipeline.compute([LAMBDABinned, ReferenceGroup]).values()
ref

input parameters

In [ ]:
file_names = [
    "TIM_ap2-2_3h_phi_157.mtz1",
    "TIM_or3_3h_phi_215.mtz1",
    "TIM_or3_3h_phi_42.mtz1",
    "TIM_ap2-2_3h_phi_66.mtz1",
    "TIM_ap2-2_3h_phi_136.mtz1",
    "TIM_or3_3h_phi_35.mtz1",
    "TIM_or3_3h_phi_222.mtz1",
    "TIM_or3_3h_phi_63.mtz1",
    "TIM_or2_3h_phi_120.mtz1",
    "TIM_or3_3h_phi_14.mtz1",
    "TIM_or2_3h_phi_141.mtz1",
    "TIM_ap2-2_3h_phi_101.mtz1",
    "TIM_or2_3h_phi_127.mtz1",
    "TIM_ap2-2_3h_phi_171.mtz1",
    "TIM_or3_3h_phi_194.mtz1",
    "TIM_ap2-2_3h_phi_192.mtz1","TIM_or3_3h_phi_229.mtz1","TIM_or3_3h_phi_28.mtz1","TIM_ap2-2_3h_phi_185.mtz1","TIM_ap2-2_3h_phi_94.mtz1","TIM_or2_3h_phi_106.mtz1","TIM_ap2-2_3h_phi_150.mtz1","TIM_or3_3h_phi_49.mtz1","TIM_or3_3h_phi_208.mtz1","TIM_ap2-2_3h_phi_108.mtz1","TIM_or2_3h_phi_92.mtz1","TIM_or3_3h_phi_180.mtz1","TIM_or3_3h_phi_07.mtz1","TIM_or2_3h_phi_113.mtz1","TIM_ap2-2_3h_phi_80.mtz1","TIM_ap2-2_3h_phi_129.mtz1","TIM_or2_3h_phi_85.mtz1","TIM_ap2-2_3h_phi_59.mtz1","TIM_ap2-2_3h_phi_164.mtz1","TIM_ap2-2_3h_phi_52.mtz1","TIM_ap2-2_3h_phi_143.mtz1","TIM_or3_3h_phi_01.mtz1","TIM_or3_3h_phi_56.mtz1","TIM_ap2-2_3h_phi_87.mtz1","TIM_or3_3h_phi_201.mtz1","TIM_ap2-2_3h_phi_122.mtz1","TIM_or3_3h_phi_21.mtz1","TIM_ap2-2_3h_phi_73.mtz1","TIM_or2_3h_phi_99.mtz1","TIM_or2_3h_phi_134.mtz1","TIM_ap2-2_3h_phi_115.mtz1","TIM_or3_3h_phi_187.mtz1","TIM_ap2-2_3h_phi_45.mtz1","TIM_ap2-2_3h_phi_178.mtz1"
]
path = "../assets/test_large/"

spacegroup_in = "C 1 2 1"  # should be a parameter
print("input spacegroup", gemmi.SpaceGroup(spacegroup_in))

# # df = mtz_to_pandas('../assets/test-permutations/together_locabs_globabs_exout_.mt.mtz')
# file = '/Users/justinbergmann/programs/LSCALE/TIM-PGA_2018/TIM_ap2-2_3h_phi_101.mtz1'
# file2 = '/Users/justinbergmann/programs/LSCALE/TIM-PGA_2018/TIM_ap2-2_3h_phi_108.mtz1'  
file = path + file_names[0]

In [ ]:
def mtz_to_pandas(file):
    ''' load mtz file into pandas dataframe '''
    mtz = gemmi.read_mtz_file(file)
    df = pd.DataFrame(data=np.array(mtz, copy=False), columns=mtz.column_labels())
    df['I_div_SIGI'] = df['I'] / df['SIGI']
    df['d'] = df.apply( lambda row: mtz.get_cell().calculate_d([int(row['H']),int(row['K']),int(row['L'])]), axis=1 )
    df['resolution'] = (1/df['d'])**2/4 # $(2d)^{-2} = \sin^2(\theta)/\lambda^2
    df.insert(0, 'hkl', df.apply(lambda row: [int(row["H"]), int(row["K"]), int(row["L"])], axis = 1))
    

    return df


def get_space_group(file, spacegroup_in):
    ''' gets spacgroup from file or uses manual addet value'''
    try:
        mtz = gemmi.read_mtz_file(file)
        print(mtz.spacegroup)
        sg = mtz.spacegroup
        asu =gemmi.ReciprocalAsu(sg)
        print("from file")
    except:
        print("manual inserted spacegroup is used")
        sg = gemmi.SpaceGroup(spacegroup_in)
        asu =gemmi.ReciprocalAsu(sg)
        print("SG",sg)
    
    print("sg2",sg)
    
    return asu, sg

def get_hkl_eq(df,asu,sg):
    mtz = gemmi.read_mtz_file(file) 
    df.insert(0, 'hkl_eq', df.apply(lambda row: asu.to_asu(row["hkl"], sg.operations())[0], axis = 1))
    # df.insert(0, 'hkl_eq', df.apply(lambda row: str(asu.to_asu(row["hkl"], sg.operations())[0]), axis = 1))
    return df


# file = '../assets/test-minimal/TIM_x3_1to1.mtz'

def read_in_files(path, file_names):
    file = path+file_names[0]
    df = mtz_to_pandas(file)
    for i in range(1,len(file_names)):
        file2 = path+file_names[i]
        df = pd.concat([df, mtz_to_pandas(file2)], ignore_index=True)
    df = df.drop('MINHARM', axis=1)
    df = df.drop('MAXHARM', axis=1)
    df = df.drop('NOVPIX', axis=1)
    df = df.drop('XF', axis=1)
    df = df.drop('YF', axis=1)
    return df



# df = df.add(mtz_to_pandas(file2))
df = read_in_files(path, file_names)
# print(df)
asu, sg = get_space_group(file, spacegroup_in)
df = get_hkl_eq(df,asu,sg)




#df = mtz_to_pandas('../assets/test-minimal/TIM_x3_4h_ap1-8_phi_01.mtz1')

df.tail()
print("number of reflexes",len(df))


In [ ]:
bins = np.linspace(df.LAMBDA.min(), df.LAMBDA.max(), 500)  # 500 is an example, you can change this, we can optimize this.
# print("wavelength bins",bins)
print("lambda min and mx",df.LAMBDA.min(), df.LAMBDA.max())
labels = ['{:.3}-{:.3}'.format(bins[i], bins[i+1]) for i in range(len(bins)-1)]
labels = range(len(bins)-1)
df['LAM_bin'] = pd.cut(df.LAMBDA, bins=bins, labels=labels)

# Find the right filter/aggregation function! .filter?


grouped = df.groupby('LAM_bin')
ref = grouped.get_group(1) # -> DataFrame
#for group_name, group_data in grouped:
    # Perform comparisons within each group
    #print(f"Group: {group_name}")
print("reflexes per bin",len(ref))
print(len(grouped),len(bins))
ref

In [ ]:
print(type(ref['hkl_eq'].iloc[0]))

get wavelength depended scale parameters 

In [ ]:
print(len(df))
ref_nr = 250
test_nr=10  # index of the data array to compare with the reference

def get_labda_scale(grouped, ref_nr, test_nr):
    """Scan through the reflexes and find the same reflexes in the reference and test data set.
    
    scale_tmp: intensity of the test dataset divided by the intensity of the reference dataset for a specific reflex
    scale_sig_tmp: the same as scale_tmp but each intensity divided by its sigma(standard uncertainty).
    """
    ref = grouped.get_group(ref_nr)
    test = grouped.get_group(test_nr) 
    scale_tmp = []
    scale_sig_tmp = []
    for i in range(len(ref)):
        
        for j in range(len(test)):
            
            # print(ref['hkl_eq'].iloc[i],  test['hkl_eq'].iloc[j])
            if ref['hkl_eq'].iloc[i] == test['hkl_eq'].iloc[j]:
                # print("ref:",ref['I'].iloc[i],"test:",test['I'].iloc[j],"It/Ir:",(test['I'].iloc[j])/(ref['I'].iloc[i]),"(It/sigt)/(Ir/sigr):",(test['I'].iloc[j]/test['SIGI'].iloc[j])/(ref['I'].iloc[i]/ref['SIGI'].iloc[i]))
                # print("equal reflex found")
                if test['I'].iloc[j]> 0 and test['SIGI'].iloc[j] > 0 and ref['I'].iloc[i] >0 and ref['SIGI'].iloc[i] >0 :
                    scale_tmp.append((test['I'].iloc[j])/(ref['I'].iloc[i]))
                    scale_sig_tmp.append((test['I'].iloc[j]/test['SIGI'].iloc[j])/(ref['I'].iloc[i]/ref['SIGI'].iloc[i]))
    
    print("scale_tmp",scale_tmp)
    print("scale_sig_tmp",scale_sig_tmp)
    scale = calculate_average(scale_tmp)
    scale_sig = calculate_average(scale_sig_tmp)
    print("scale and sacale_sig",scale,scale_sig)
    return scale, scale_sig
        
def calculate_average(lst):
    if not lst:  # Check if the list is empty
        return None  # Return None for an empty list
    return sum(lst) / len(lst)

scale, scale_sig = get_labda_scale(grouped,ref_nr,test_nr)
print("scale, scale_sig",scale, scale_sig)


find the scale factor for each wavelength

In [ ]:
# The part that should speed up.

ref_nr = int(len(grouped)/2)
print("ref",ref_nr)
l_scale = []
l_scale_sig =[]
l_lambda = []
for i in range(len(grouped)/4,3*len(grouped)/4):
    scale, scale_sig = get_labda_scale(grouped,ref_nr,i)
    l_scale.append(scale)
    l_scale_sig.append(scale_sig)
    l_lambda.append((bins[i]+bins[i+1])/2)
    print(i,(bins[i]+bins[i+1])/2 )




remove outliers with more than x sigma divergence.

In [ ]:
def filter_grouped(lam,data,lim):
    qq= np.sqrt(np.sqrt(data))
    std_dev = np.std(qq)
    average = np.mean(qq)
    print(average,std_dev)

    f_lam=[]
    f_sca = []
    for i in range(len(qq)):
        if qq[i] > average - lim*std_dev and qq[i]< average + lim*std_dev:
            f_lam.append(lam[i])
            f_sca.append(data[i])
    return f_lam,f_sca


l_lam_f, l_sca_f=filter_grouped(l_lambda,l_scale,2)
print(len(l_lam_f))

# l_lam_f, l_sca_f=filter_grouped(l_lam_f, l_sca_f,3)


In [ ]:
data = l_sca_f
# data = l_scale


# Create density plot
plt.hist(data, bins=100, density=True, alpha=0.5, color='g')
plt.title('Density Plot ')
plt.xlabel('Value')
plt.ylabel('Density')
plt.show()

In [ ]:
data = np.sqrt(np.sqrt( l_sca_f))
# data = np.sqrt(np.sqrt( l_scale))

# Create density plot
plt.hist(data, bins=100, density=True, alpha=0.5, color='g')
plt.title('Density Plot ')
plt.xlabel('Value')
plt.ylabel('Density')
plt.show()

In [ ]:
import scipy.stats as stats
data = np.sqrt(np.sqrt( l_sca_f))
# data = np.sqrt(np.sqrt( l_scale))
fig, ax = plt.subplots()
stats.probplot(data, dist="norm", plot=ax)
ax.set_title('Q-Q Plot')
plt.show()

In [ ]:
x= l_lambda
y = l_scale
x = l_lam_f
y = l_sca_f

# Fit a polynomial curve (in this case, a linear curve)
coefficients = np.polyfit(x, y, deg=7)
poly_function = np.poly1d(coefficients)


# Generate points for the fitted curve
x_fit = np.linspace(min(x), max(x), len(x))
y_fit = poly_function(x_fit)

norm_coeff = np.array([1, -1, 1])
chebyshev = np.polynomial.chebyshev.Chebyshev(norm_coeff)
l = x
scale_function = np.vectorize(chebyshev / chebyshev(l_lambda[ref_nr]) )
plt.plot(l, scale_function(l),label='chebyshev', color='g')






# Plot original data and fitted curve
plt.scatter(x, y, label='Data')
plt.plot(x_fit, y_fit, color='red', label='Fitted Curve')
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()
plt.ylim(0,4)
plt.show()
print(poly_function)
print(len(x)-len(x_fit))
print(poly_function(3))
print("chebyshev",scale_function(3))

In [ ]:
from scipy.optimize import curve_fit

# Example data
x = np.array(x)
y = np.array(y)

# Exponential function to fit
def exponential_func(x, a, b):
    return a * np.exp(b * x)

def pol_func(x , a,b,c,d,e,f,g,h):
    return a + b*x + c*x**2 +d*x**3 + e*x**4 + f*x**5 + g*x**6 + h*x**7

def chep_func(x, *params):
    # a, b = params
    chebyshev = np.polynomial.chebyshev.Chebyshev(params)
    # print("chebyshev",chebyshev)
    return chebyshev(x)

# Fit the data to the exponential function
initial_guess = np.ones(7) # Initial guess for parameters
params, covariance = curve_fit(chep_func, x, y, p0=initial_guess)

# Fit the data to the exponential function
# params, covariance = curve_fit(exponential_func, x, y)
# params, covariance = curve_fit(pol_func, x, y)
# Plot the original data and the fitted curve
plt.scatter(x, y, label='Data')
# plt.plot(x, exponential_func(x, *params), color='red', label='Fitted Curve')
plt.plot(x, chep_func(x, *params), color='red', label='chep_func')
# plt.plot(x, pol_func(x, *params), color='red', label='Fitted Curve')
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()
plt.show()
print("params",params,"covariance",covariance)

In [ ]:
# plt.scatter(x, y, label='Data')
# plt.plot(x, exponential_func(x, *params), color='red', label='Fitted Curve')

plt.plot(x, chep_func(x, *params)/chep_func(l_lambda[ref_nr], *params), color='red', label='chep_func')
# plt.plot(x, pol_func(x, *params), color='red', label='Fitted Curve')
plt.xlabel('wavelength')
plt.ylabel('scale factor')
plt.legend()
plt.show()
print("params",params,"covariance",covariance)

In [ ]:
def get_lambda_scale(df,scale_function,*params):
    df.insert(len(df.columns), 'lambda_scale', df.apply(lambda row: scale_function(row["LAMBDA"]), axis = 1))
    return df

def apply_scales(df):
   df.insert(len(df.columns), 'scaled_I', df.apply(lambda row: row["I"]*row["lambda_scale"], axis = 1))
   df.insert(len(df.columns), 'scaled_SIGI', df.apply(lambda row: row["SIGI"]*row["lambda_scale"], axis = 1)) 
   return df 


In [ ]:
df = get_lambda_scale(df,scale_function)
df = apply_scales(df)

In [ ]:
df = apply_scales(df)


In [ ]:
df

In [ ]:
write_out = path+"scaled.mtz"
write_out 
df1.write_to_file(write_out)

In [ ]:
from scipy.optimize import curve_fit

# Example data
x = np.array(x)
y = np.array(y)

# Exponential function to fit
def exponential_func(x, a, b):
    return a * np.exp(b * x)

def pol_func(x , a,b,c,d,e,f,g,h):
    return a + b*x + c*x**2 +d*x**3 + e*x**4 + f*x**5 + g*x**6 + h*x**7

def chep_func(x, *params):
    # a, b = params
    chebyshev = np.polynomial.chebyshev.Chebyshev(params)
    # print("chebyshev",chebyshev)
    return chebyshev(x)

# Fit the data to the exponential function
initial_guess = np.ones(7) # Initial guess for parameters
params, covariance = curve_fit(chep_func, x, y, p0=initial_guess)

# Fit the data to the exponential function
# params, covariance = curve_fit(exponential_func, x, y)
# params, covariance = curve_fit(pol_func, x, y)
# Plot the original data and the fitted curve
plt.scatter(x, y, label='Data')
# plt.plot(x, exponential_func(x, *params), color='red', label='Fitted Curve')
plt.plot(x, chep_func(x, *params), color='red', label='chep_func')
# plt.plot(x, pol_func(x, *params), color='red', label='Fitted Curve')
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()
plt.show()
print("params",params,"covariance",covariance)


In [ ]:
# plt.scatter(x, y, label='Data')
# plt.plot(x, exponential_func(x, *params), color='red', label='Fitted Curve')
plt.plot(x, chep_func(x, *params)/chep_func(3, *params), color='red', label='chep_func')
# plt.plot(x, pol_func(x, *params), color='red', label='Fitted Curve')
plt.xlabel('wavelength')
plt.ylabel('scale factor')
plt.legend()
plt.show()
print("params",params,"covariance",covariance)

In [ ]:
norm_coeff = np.array([1, -1, 1])  
chebyshev = np.polynomial.chebyshev.Chebyshev(norm_coeff)
print(chebyshev(2))

In [ ]:
#get weights
x = l_lam_f
y = l_sca_f
l_weight= []
for i in range(len(y)):
    weight = ((poly_function(x[i])-y[i])**2)
    l_weight.append(weight)
print(l_weight)
    

In [ ]:
# plt.scatter(l_lambda, l_scale,label='scale')
# plt.scatter(l_lambda, l_scale_sig,label='scale_sig')
plt.scatter(l_lam_f, l_sca_f,label='scale')
plt.xlabel(r'$\lambda$')
plt.ylabel(r'$scale$')
plt.title('Lambda scaling ')
plt.ylim(0,4)
plt.legend()
plt.plot()

In [ ]:

def smoothify(thisarray):
    """
    returns moving average of input using:
    out(n) = .7*in(n) + 0.15*( in(n-1) + in(n+1) )
    """

    # make sure we got a numpy array, else make it one
    if type(thisarray) == type([]): thisarray = np.array(thisarray)

    # do the moving average by adding three slices of the original array
    # returns a numpy array,
    # could be modified to return whatever type we put in...
    return 0.7 * thisarray[1:-1] + 0.15 * ( thisarray[2:] + thisarray[:-2] )

myarray = [1, 20, 55, 33, 4555555, 1]
smooth_l_scale = smoothify(l_scale)
smooth_l_lambda = smoothify(l_lambda)
# smooth_l_scale_sig = smoothify(l_scale_sig)
print(len(l_scale),len(smooth_l_scale))
plt.scatter(l_lam_f, l_sca_f,label='org')
plt.scatter(smooth_l_lambda, smooth_l_scale,label='scale')
# plt.scatter(smooth_l_lambda, smooth_l_scale_sig,label='scale_sig')
plt.xlabel(r'$\lambda$')
plt.ylabel(r'$scale$')
plt.title('Lambda scaling ')
plt.ylim(0,4)
plt.legend()
plt.plot()

In [ ]:
def residual_weighted_smoothing(data, weights):
    if len(data) != len(weights):
        raise ValueError("Lengths of data and weights must be the same")

    smoothed_data = [data[0]]  # Initialize with the first data point
    for i in range(1, len(data)):
        residual = data[i] - smoothed_data[i - 1]  # Calculate the residual
        smoothed_value = smoothed_data[i - 1] + weights[i] * residual  # Smooth the residual
        smoothed_data.append(smoothed_value)
    return smoothed_data

# Example usage:
data = [10, 20, 30, 40, 50]
weights = [0.2, 0.3, 0.4, 0.5, 0.6]
x= [1,2,3,4,5]

smoothed_data = residual_weighted_smoothing(l_sca_f, l_weight)
print("Smoothed data:", smoothed_data)

plt.scatter(l_lam_f, l_sca_f,label='org')
plt.scatter(l_lam_f,smoothed_data,label='smoothing')
# plt.ylim(0,4)
plt.legend()
plt.plot()


In [ ]:
def weighted_smoothing(data, weights):
    if len(data) != len(weights):
        raise ValueError("Lengths of data and weights must be the same")
    
    smoothed_data = []
    for i in range(len(data)):
        if i == 0:
            smoothed_data.append(data[0])
        else:
            smoothed_value = (data[i] * weights[i] + smoothed_data[i - 1] * (1 - weights[i])) / (1 + weights[i])
            smoothed_data.append(smoothed_value)
    return smoothed_data

# Example usage:
data = [10, 20, 30, 40, 50]
weights = [0.2, 0.3, 0.4, 0.5, 0.6]


smoothed_data = residual_weighted_smoothing(l_sca_f, l_weight)
# print("Smoothed data:", smoothed_data)

plt.scatter(l_lam_f, l_sca_f,label='org')
plt.scatter(l_lam_f,smoothed_data,label='smoothed')
# plt.ylim(0,6)
plt.legend()
plt.plot()
print("smoothed_data",smoothed_data)

In [ ]:
# Generate example data


In [ ]:
plt.hist(df['I'], bins=10, density=False)
#plt.ylim(0,100)

In [ ]:
plt.plot(df['resolution'], df['I_div_SIGI'], 'k.', ms=1, alpha=0.8)
plt.xlabel(r'$(2d)^{-2} = \sin^2(\theta)/\lambda^2$')
plt.ylabel(r'$I/\sigma(I)$')
plt.title('LSCALE: I/sigma(I) versus resolution')
plt.plot()

In [ ]:
plt.plot(df['LAM'], df['resolution'], 'k.', ms=1, alpha=0.8)
plt.title('LSCALE: Resolution vs. lambda')
plt.xlabel('$\lambda$')
plt.ylabel('resolution')

In [ ]:
df.plot(x='LAM', y='I_div_SIGI', kind='scatter', marker='.', s=1, alpha=0.5)
plt.title('LSCALE: I/sigma(I) vs. lambda')
plt.plot()

In [ ]:
df.plot(x='LAM', y='d', kind='scatter', marker='.', s=1, alpha=0.5)
plt.xlabel(r'$\lambda$')
plt.ylabel(r'$d$')
plt.plot()

In [ ]:
# serial_out_min/together_para_plate_imagesc_imagebf_norm_.log
#NORM_COEFF_1 10 2.730 3.540 2.00000e+00 0.00000e+00 0.00000e+00 0.00000e+00 0.00000e+00 0.00000e+00 0.00000e+00 0.00000e+00 0.00000e+00 0.00000e+00 0.00000e+00
norm_coeff = np.array([1, -1, 1])
chebyshev = np.polynomial.chebyshev.Chebyshev(norm_coeff)
l = np.linspace(-1, 1, 100)
f = np.vectorize(chebyshev / chebyshev(3.2))
plt.plot(l, f(l))
#chebyshev(3.2)

In [ ]:
norm_coeff[::-1]

# Compare multiple MTZ files

In [ ]:
mtz_files = glob.glob('../assets/test-permutations/*.mtz')
mtz_files, len(mtz_files)

In [ ]:
fig, axs = plt.subplots(1,4, sharex=True, sharey=True)
current_axis = 0
for file in mtz_files[:4]:
    df = mtz_to_pandas(file)
    df.plot(x='LAM', y='I_div_SIGI', kind='scatter', marker='.', s=1, alpha=0.5, ax=axs[current_axis])
    title = Path(file).stem
    axs[current_axis].set_title(title, fontsize=4)
    current_axis += 1
    